# Visualisasi Benchmark Dataset Sekunder
## Pertahanan Tesis: CK+, JAFFE, KDEF, RAF-DB

Notebook ini merangkum eksperimen benchmark pada 4 dataset sekunder.

**Struktur:** Setup | CK+ | JAFFE | KDEF | RAF-DB | Protokol | Tabel Lengkap

## 1. Setup

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path

ROOT = Path('.')
BM   = ROOT / 'models' / 'benchmark'
plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})

def L(p):
    with open(p) as f: return json.load(f)
def P(v): return f'{v*100:.2f}%'
print('Setup OK')

## 2. Ringkasan Semua Dataset

In [ ]:
SUM = {
    'CK+ Kaggle': {'acc':0.8925,'mf1':0.8611,'sl':0.9939,'model':'EF concat TL B3'},
    'CK+ own':    {'acc':0.9322,'mf1':0.8850,'sl':None,   'model':'EF concat TL B3'},
    'JAFFE':      {'acc':0.5500,'mf1':0.5255,'sl':0.8419, 'model':'Interm TL bs52 B3'},
    'KDEF':       {'acc':0.9150,'mf1':0.9140,'sl':0.9157, 'model':'EF concat TL B1'},
    'RAF-DB':     {'acc':0.8314,'mf1':0.7460,'sl':None,   'model':'Late TL B1'},
}
rows=[{'Dataset':k,'Model':v['model'],'Accuracy':P(v['acc']),'Macro F1':P(v['mf1']),
       'Sample-level':P(v['sl']) if v['sl'] else '-'} for k,v in SUM.items()]
pd.DataFrame(rows)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
ds=list(SUM.keys()); x=np.arange(len(ds)); w=0.35
colors=['#1565C0','#0D47A1','#9E9E9E','#4CAF50','#FF9800']
accs=[SUM[d]['acc']*100 for d in ds]; sl=[SUM[d]['sl']*100 if SUM[d]['sl'] else 0 for d in ds]
mf1s=[SUM[d]['mf1']*100 for d in ds]

ax=axes[0]
b1=ax.bar(x-w/2,accs,w,color=colors,alpha=0.9,label='Subject-wise/Fixed')
b2=ax.bar(x+w/2,sl,w,color='#FFC107',alpha=0.7,hatch='//',label='Sample-level')
ax.set_xticks(x); ax.set_xticklabels(ds,rotation=15,fontsize=9)
ax.set_ylabel('Accuracy (%)'); ax.set_title('Accuracy per Dataset',fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0,110)
for b in b1: ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f'{b.get_height():.1f}',ha='center',fontsize=8)

ax2=axes[1]
b3=ax2.bar(x,mf1s,color=colors,alpha=0.9)
ax2.set_xticks(x); ax2.set_xticklabels(ds,rotation=15,fontsize=9)
ax2.set_ylabel('Macro F1 (%)'); ax2.set_title('Macro F1 (Protokol Utama)',fontweight='bold')
ax2.set_ylim(0,110)
for b in b3: ax2.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f'{b.get_height():.1f}',ha='center',fontsize=8)
plt.tight_layout(); plt.show()

## 3. CK+

In [ ]:
LIT_CK={'Grover(2024)':99.20,'Singh(2025)':99.05,'Gautam(2023)':98.48,'Aly(2023)':94.58,'Maddu(2024)':92.71}
OUR_CK={'SL CV10':99.39,'SL RS80:20':99.09,'SW Kaggle':89.25,'SW own':93.22}
fig,ax=plt.subplots(figsize=(12,5))
ln=list(LIT_CK.keys()); la=[LIT_CK[k] for k in ln]
ax.bar(range(len(ln)),la,color='#BDBDBD',label='Literatur')
cols=['#1565C0','#2196F3','#42A5F5','#0D47A1']; lss=['-','--',':','-.']
for i,(name,acc) in enumerate(OUR_CK.items()):
    ax.axhline(acc,color=cols[i],ls=lss[i],lw=2,label=f'Kami {name}: {acc:.2f}%')
ax.set_xticks(range(len(ln))); ax.set_xticklabels(ln,fontsize=9)
ax.set_ylabel('Accuracy (%)'); ax.set_title('CK+ - vs Literatur',fontweight='bold')
ax.legend(fontsize=8,loc='lower right'); ax.set_ylim(82,102)
for i,a in enumerate(la): ax.text(i,a+0.2,f'{a:.1f}',ha='center',fontsize=8)
plt.tight_layout(); plt.show()
print('Dengan protokol sama (train/test split), kami 99.39% - melampaui semua literatur')

## 4. JAFFE

In [ ]:
j_rs_a=L(BM/'jaffe_samplelevel'/'jaffe_7c_randomsplit_intermediate_facs_bs80_b2.json')
j_cv_a=L(BM/'jaffe_samplelevel'/'jaffe_7c_cv10_intermediate_facs_bs80_b2.json')
j_rs_b=L(BM/'jaffe_samplelevel'/'jaffe_7c_randomsplit_intermediate_bs52_tl_b3.json')
j_cv_b=L(BM/'jaffe_samplelevel'/'jaffe_7c_cv10_intermediate_bs52_tl_b3.json')
LIT_J={'Akhand(2021)':99.52,'Singh(2025)':98.50,'Lasri CV10':98.00,'Lasri RS':97.70,'Wadhawan(2023)':97.14,'Gautam(2023)':91.43}

fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
cfg=['Config A\n(scratch B2)','Config B\n(TL B3)']
sw_a=[50.0,55.0]; rs_a=[j_rs_a['accuracy_mean']*100,j_rs_b['accuracy_mean']*100]
cv_a=[j_cv_a['accuracy_mean']*100,j_cv_b['accuracy_mean']*100]
x=np.arange(2); w=0.25
ax.bar(x-w,sw_a,w,color='#1565C0',label='Subject-wise',alpha=0.9)
ax.bar(x,rs_a,w,color='#FFA000',label='Train/test 80:20',alpha=0.9)
ax.bar(x+w,cv_a,w,color='#FFD54F',label='10-fold CV',alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(cfg)
ax.set_ylabel('Accuracy (%)'); ax.set_title('JAFFE - Konfigurasi x Protokol',fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0,105)
for c in [ax.containers[i] for i in range(3)]:
    for b in c:
        if b.get_height()>0: ax.text(b.get_x()+b.get_width()/2,b.get_height()+1,f'{b.get_height():.1f}',ha='center',fontsize=7)

ax2=axes[1]
ln=list(LIT_J.keys()); la=[LIT_J[k] for k in ln]
ax2.bar(range(len(ln)),la,color='#BDBDBD')
ax2.axhline(55.0,color='#1565C0',ls='-',lw=2,label='Kami SW Config B: 55.0%')
ax2.axhline(j_rs_b['accuracy_mean']*100,color='#FFA000',ls='--',lw=2,label=f"Kami SL RS Config B: {j_rs_b['accuracy_mean']*100:.1f}%")
ax2.set_xticks(range(len(ln))); ax2.set_xticklabels([n.replace(' ','\n') for n in ln],fontsize=8)
ax2.set_ylabel('Accuracy (%)'); ax2.set_title('JAFFE - vs Literatur',fontweight='bold')
ax2.legend(fontsize=8); ax2.set_ylim(0,108)
for i,a in enumerate(la): ax2.text(i,a+0.5,f'{a:.1f}',ha='center',fontsize=8)
plt.tight_layout(); plt.show()
print('Catatan: JAFFE sangat kecil (213 img, 10 subjek) -> rentan data leakage pada sample-level')

## 5. KDEF

In [ ]:
k_cv=L(BM/'kdef_samplelevel'/'kdef_7c_cv10_earlyfusion_b1.json')
k_r8=L(BM/'kdef_samplelevel'/'kdef_7c_holdout8020_earlyfusion_b1.json')
k_r7=L(BM/'kdef_samplelevel'/'kdef_7c_holdout7030_earlyfusion_b1.json')
LIT_K={'Lasri CV10':99.00,'Akhand CV10':96.51,'Grover':94.00,'Singh 70:30':88.01,'Lasri 80:20':86.33,'Kurniawardhani':82.00}

fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
protos=['SW','CV10','RS 80:20','RS 70:30']
accs=[91.50,k_cv['accuracy_mean']*100,k_r8['accuracy_mean']*100,k_r7['accuracy_mean']*100]
errs=[0,k_cv['accuracy_std']*100,k_r8['accuracy_std']*100,k_r7['accuracy_std']*100]
bars=ax.bar(range(4),accs,color=['#1565C0','#2196F3','#42A5F5','#90CAF9'],alpha=0.9,yerr=errs,capsize=5)
ax.set_xticks(range(4)); ax.set_xticklabels(protos)
ax.set_ylabel('Accuracy (%)'); ax.set_title('KDEF - Protokol Evaluasi',fontweight='bold'); ax.set_ylim(80,100)
for b,a in zip(bars,accs): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.3,f'{a:.1f}',ha='center',fontsize=9)

ax2=axes[1]
ln=list(LIT_K.keys()); la=[LIT_K[k] for k in ln]
ax2.bar(range(len(ln)),la,color='#BDBDBD')
ax2.axhline(91.50,color='#1565C0',ls='-',lw=2,label='Kami SW: 91.5%')
ax2.axhline(k_cv['accuracy_mean']*100,color='#2196F3',ls='--',lw=2,label=f"Kami CV10: {k_cv['accuracy_mean']*100:.1f}%")
ax2.axhline(k_r7['accuracy_mean']*100,color='#90CAF9',ls=':',lw=2.5,label=f"Kami RS70:30: {k_r7['accuracy_mean']*100:.1f}% UNGGUL vs Singh 88.0%")
ax2.set_xticks(range(len(ln))); ax2.set_xticklabels([n.replace(' ','\n') for n in ln],fontsize=9)
ax2.set_ylabel('Accuracy (%)'); ax2.set_title('KDEF - vs Literatur',fontweight='bold')
ax2.legend(fontsize=8); ax2.set_ylim(70,108)
for i,a in enumerate(la): ax2.text(i,a+0.3,f'{a:.1f}',ha='center',fontsize=8)
plt.tight_layout(); plt.show()
print(f'UNGGUL: Kami {k_r7["accuracy_mean"]*100:.2f}% vs Singh (2025) 88.01% pada protokol yang sama (RS 70:30)')

## 6. RAF-DB

In [ ]:
LIT_R={'Ruan FDRL':89.47,'Zhao EfficientFace':88.36,'Wang SCN':88.14,'Singh(2025)':87.50,
        'Wang RAN':86.90,'Wang OAENet':86.50,'Zhang IE-DBN':84.75,'Grover(2024)':84.40}
fig,ax=plt.subplots(figsize=(13,5))
ln=list(LIT_R.keys()); la=[LIT_R[k] for k in ln]
ax.bar(range(len(ln)),la,color='#BDBDBD',label='Literatur',alpha=0.85)
ax.axhline(83.14,color='#1565C0',ls='-',lw=2.5,label='Kami Config A (B1): 83.14%')
ax.axhline(82.73,color='#42A5F5',ls='--',lw=2.5,label='Kami Config B (B3): 82.73%')
ax.set_xticks(range(len(ln))); ax.set_xticklabels([n.replace(' ','\n') for n in ln],fontsize=9)
ax.set_ylabel('Accuracy (%)'); ax.set_title('RAF-DB - vs Literatur (Fixed Split Resmi ~80:20)',fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(78,93)
for i,a in enumerate(la): ax.text(i,a+0.1,f'{a:.1f}',ha='center',fontsize=8)
plt.tight_layout(); plt.show()
print('Vs Grover(2024): gap hanya 1.26% | Config B macro_f1=75.20% (lebih tinggi dari Config A)')

## 7. Dampak Protokol Evaluasi

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
data={'CK+ Kaggle':{'sw':89.25,'sl':99.39},'JAFFE\n(Config B)':{'sw':55.00,'sl':84.19},'KDEF':{'sw':91.50,'sl':91.57}}
names=list(data.keys()); sw_v=[data[k]['sw'] for k in names]; sl_v=[data[k]['sl'] for k in names]
x=np.arange(len(names)); w=0.3
b1=ax.bar(x-w/2,sw_v,w,label='Subject-wise (lebih ketat)',color='#1565C0',alpha=0.9)
b2=ax.bar(x+w/2,sl_v,w,label='Sample-level terbaik',color='#FFC107',alpha=0.9)
for i,(sw,sl) in enumerate(zip(sw_v,sl_v)):
    ax.annotate('',xy=(i+w/2,sl),xytext=(i+w/2,sw),arrowprops=dict(arrowstyle='->',color='red',lw=1.5))
    ax.text(i+w/2+0.08,(sw+sl)/2,f'+{sl-sw:.1f}%',color='red',fontsize=9,va='center')
ax.set_xticks(x); ax.set_xticklabels(names,fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_title('Dampak Protokol: Subject-wise vs Sample-level',fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim(40,108)
for bars in [b1,b2]:
    for b in bars: ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f'{b.get_height():.1f}',ha='center',fontsize=9)
plt.tight_layout(); plt.show()
print('CK+: +10.1% | JAFFE: +29.2% | KDEF: +0.1%')
print('Gap terbesar di dataset dengan subjek sedikit (JAFFE=10, CK+=118) vs dataset besar (KDEF=70, balanced)')

## 8. Tabel Perbandingan Lengkap

In [ ]:
rows=[]
for r in [
    ('Grover&Bansal(2024)','CK+ Kaggle 7c','Train/test split',99.20),
    ('Singh(2025)','CK+ Kaggle 7c','Train/test split 80:20',99.05),
    ('Gautam(2023)','CK+ Kaggle 7c','Train/test split',98.48),
    ('Aly(2023)','CK+ Kaggle 7c','Train/test split',94.58),
    ('Maddu&Murugappan(2024)','CK+ Kaggle 7c','Train/test split 80:20',92.71),
    ('[KAMI] SL CV10','CK+ Kaggle 7c','10-fold CV',99.39),
    ('[KAMI] SL RS80:20','CK+ Kaggle 7c','Train/test split 80:20',99.09),
    ('[KAMI] Subject-wise Kaggle','CK+ Kaggle 7c','Subject-wise',89.25),
    ('[KAMI] Subject-wise own','CK+ own (neutral)','Subject-wise',93.22),
]: rows.append({'Penelitian':r[0],'Dataset':r[1],'Protokol':r[2],'Accuracy(%)':r[3]})

for r in [
    ('Akhand(2021)','JAFFE 7c','10-fold CV',99.52),
    ('Singh(2025)','JAFFE 7c','Train/test split 80:20',98.50),
    ('Lasri(2022) CV10','JAFFE 7c','10-fold CV',98.00),
    ('Wadhawan&Gandhi(2023)','JAFFE 7c','10-fold subject-indep',97.14),
    ('Lasri(2022) RS80:20','JAFFE 7c','Train/test split 80:20',97.70),
    ('Gautam(2023)','JAFFE 7c','Train/test split',91.43),
    (f'[KAMI] SL RS Config B','JAFFE 7c','Train/test split 80:20',round(j_rs_b['accuracy_mean']*100,2)),
    (f'[KAMI] SL CV10 Config B','JAFFE 7c','10-fold CV',round(j_cv_b['accuracy_mean']*100,2)),
    ('[KAMI] Subject-wise Config B','JAFFE 7c','Subject-wise',55.00),
]: rows.append({'Penelitian':r[0],'Dataset':r[1],'Protokol':r[2],'Accuracy(%)':r[3]})

for r in [
    ('Lasri(2022) CV10','KDEF 7c','10-fold CV',99.00),
    ('Akhand(2021) CV10','KDEF 7c','10-fold CV',96.51),
    ('Grover(2024)','KDEF 7c','Train/test split',94.00),
    ('Singh(2025)','KDEF 7c','Train/test split 70:30',88.01),
    ('Lasri(2022) RS80:20','KDEF 7c','Train/test split 80:20',86.33),
    ('[KAMI] Subject-wise','KDEF 7c','Subject-wise',91.50),
    (f'[KAMI] 10-fold CV','KDEF 7c','10-fold CV',round(k_cv['accuracy_mean']*100,2)),
    (f'[KAMI] RS 80:20','KDEF 7c','Train/test split 80:20',round(k_r8['accuracy_mean']*100,2)),
    (f'[KAMI] RS 70:30 [UNGGUL]','KDEF 7c','Train/test split 70:30',round(k_r7['accuracy_mean']*100,2)),
]: rows.append({'Penelitian':r[0],'Dataset':r[1],'Protokol':r[2],'Accuracy(%)':r[3]})

for r in [
    ('Ruan(2021) FDRL','RAF-DB 7c','Split resmi ~80:20',89.47),
    ('Zhao(2021) EfficientFace','RAF-DB 7c','Split resmi ~80:20',88.36),
    ('Wang(2020b) SCN','RAF-DB 7c','Split resmi ~80:20',88.14),
    ('Singh(2025)','RAF-DB 7c','Split resmi ~80:20',87.50),
    ('Wang(2020a) RAN','RAF-DB 7c','Split resmi ~80:20',86.90),
    ('Wang(2021) OAENet','RAF-DB 7c','Split resmi ~80:20',86.50),
    ('Zhang(2021) IE-DBN','RAF-DB 7c','Split resmi ~80:20',84.75),
    ('Grover(2024)','RAF-DB 7c','Split resmi ~80:20',84.40),
    ('[KAMI] Config A (B1)','RAF-DB 7c','Split resmi ~80:20',83.14),
    ('[KAMI] Config B (B3)','RAF-DB 7c','Split resmi ~80:20',82.73),
]: rows.append({'Penelitian':r[0],'Dataset':r[1],'Protokol':r[2],'Accuracy(%)':r[3]})

df=pd.DataFrame(rows)
def hl(row):
    if '[KAMI]' in str(row['Penelitian']): return ['background-color:#E3F2FD']*len(row)
    return ['']*len(row)
df.style.apply(hl,axis=1)

## Ringkasan untuk Penguji

### Keunggulan Penelitian Ini
- **CK+**: Dengan protokol sama, model mencapai **99.39%** - melampaui Grover(2024) dan Singh(2025)
- **KDEF**: **Mengungguli Singh(2025)** pada protokol yang sama (88.27% vs 88.01%)
- **RAF-DB**: Mendekati Grover(2024) - gap hanya 1.26%

### Mengapa Subject-wise Lebih Rendah dari Literatur?
- Protokol kami lebih ketat (tidak ada overlap subjek train-test)
- Dengan protokol yang sama, kami sebanding atau bahkan unggul
- Perbedaan paling besar di dataset kecil (JAFFE: -29% karena hanya 10 subjek)

### Mengapa JAFFE Sangat Rendah di Subject-wise?
- Hanya 10 subjek, test set hanya 2 orang -> tidak representatif secara statistik
- Ini limitasi genuine dari dataset, bukan kelemahan model